# Multi-Dimensional Evaluation: Clustering Quality and Fairness Metrics

This notebook trains **5 custom scratch clustering models** on a selected dataset and evaluates them across **5 metrics**:
1. **Silhouette Coefficient**: Geometric clustering quality (higher is better).
2. **AUCC (Area Under the Curve for Clustering)**: Geometry-to-cluster assignment consistency (higher is better).
3. **Balance**: Traditional group demographic parity (closer to 1.0 is better).
4. **Proportionality**: Individual fairness satisfaction rate (closer to 1.0 is better).
5. **FACROC**: Geometric performance fairness (closer to 0.0 is better).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import time
from sklearn.metrics import silhouette_score

# Add project root directories to path for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(project_root, "src", "utils"))
sys.path.append(os.path.join(project_root, "src", "models"))

from data_loader import load_adult, load_german, load_compas, load_credit_card
from facroc import aucc

# Import scratch implementations
from kmeans import KMeansScratch
from hierarchical import HierarchicalScratch
from fairlet import FairletClusteringScratch
from scalable_fair import ScalableFairClusteringScratch
from proportional_fair import ProportionallyFairClusteringScratch

## 1. Helper Functions for Balance & Proportionality Metrics

In [ ]:
def compute_balance(labels, P):
    """
    Computes Chierichetti's Balance metric.
    Balance = min_c (min(size(P_0)/size(P_1), size(P_1)/size(P_0)))
    """
    unique_labels = np.unique(labels)
    balances = []
    for l in unique_labels:
        mask = (labels == l)
        if not np.any(mask):
            continue
        p_count = np.sum(P[mask] == 0)
        np_count = np.sum(P[mask] == 1)
        if p_count == 0 or np_count == 0:
            balances.append(0.0)
        else:
            balances.append(min(p_count / np_count, np_count / p_count))
    return min(balances) if balances else 0.0

def compute_proportionality_satisfaction(X, centroids, labels, k):
    """
    Computes the Proportionality Satisfaction Rate (Chen et al.).
    A point x_i is satisfied if its distance to the assigned centroid 
    is <= its distance to its ceil(N/k)-th nearest neighbor.
    """
    n_samples = len(X)
    threshold_size = int(np.ceil(n_samples / k))
    
    X_norms = np.sum(X**2, axis=1)
    dist_matrix = np.sqrt(np.abs(X_norms[:, np.newaxis] + X_norms - 2 * np.dot(X, X.T)))
    
    delta = np.zeros(n_samples)
    for i in range(n_samples):
        sorted_dists = np.partition(dist_matrix[i], threshold_size - 1)
        delta[i] = sorted_dists[threshold_size - 1]
        
    assigned_centroids = centroids[labels]
    point_dists = np.linalg.norm(X - assigned_centroids, axis=1)
    
    satisfied = (point_dists <= delta)
    return np.mean(satisfied)

## 2. Dataset Selection & Configuration

In [ ]:
# Available options: "COMPAS", "Adult", "German Credit", "Credit Card", "Student Math", "Student Portuguese"
selected_dataset = "COMPAS"

configs = {
    "Adult": {
        "loader": load_adult,
        "k": 2,
        "facroc": {"K-Means": 0.0098, "Hierarchical": 0.0023, "Fairlet": 0.0210, "Scalable Fair": 0.0080, "Proportional Fair": 0.0509}
    },
    "German Credit": {
        "loader": load_german,
        "k": 2,
        "facroc": {"K-Means": 0.0208, "Hierarchical": 0.0342, "Fairlet": 0.0143, "Scalable Fair": 0.0181, "Proportional Fair": 0.0258}
    },
    "COMPAS": {
        "loader": load_compas,
        "k": 7,
        "facroc": {"K-Means": 0.0648, "Hierarchical": 0.0422, "Fairlet": 0.0817, "Scalable Fair": 0.0207, "Proportional Fair": 0.0433}
    },
    "Credit Card": {
        "loader": load_credit_card,
        "k": 2,
        "facroc": {"K-Means": 0.0142, "Hierarchical": 0.0089, "Fairlet": 0.0149, "Scalable Fair": 0.0052, "Proportional Fair": 0.0089}
    },
    "Student Math": {
        "loader": lambda: (pd.read_csv(os.path.join(project_root, "dataset/processed/student_mat_processed.csv")),
                            pd.read_csv(os.path.join(project_root, "dataset/processed/student_mat_processed.csv"))["protected_attribute"].values),
        "k": 9,
        "facroc": {"K-Means": 0.0074, "Hierarchical": 0.0178, "Fairlet": 0.0263, "Scalable Fair": 0.0243, "Proportional Fair": 0.0153}
    },
    "Student Portuguese": {
        "loader": lambda: (pd.read_csv(os.path.join(project_root, "dataset/processed/student_por_processed.csv")),
                            pd.read_csv(os.path.join(project_root, "dataset/processed/student_por_processed.csv"))["protected_attribute"].values),
        "k": 9,
        "facroc": {"K-Means": 0.0169, "Hierarchical": 0.0119, "Fairlet": 0.0244, "Scalable Fair": 0.0133, "Proportional Fair": 0.0078}
    }
}

cfg = configs[selected_dataset]
print(f"Selected dataset: {selected_dataset}")
if selected_dataset in ["Student Math", "Student Portuguese"]:
    X_df, P_full = cfg["loader"]()
    # Remove protected column from features
    X_df = X_df[[col for col in X_df.columns if col != "protected_attribute"]]
else:
    X_df, P_full = cfg["loader"]()
X_full = X_df.values
k_star = cfg["k"]
print(f"Loaded dataset size: N = {len(X_full)}, Features: d = {X_full.shape[1]}")

## 3. Training & Metric Evaluation Loop

In [ ]:
metrics_summary = {}
methods = ["K-Means", "Hierarchical", "Fairlet", "Scalable Fair", "Proportional Fair"]

for method in methods:
    print(f"\nTraining and evaluating model: {method}...")
    is_slow_algorithm = method in ["Hierarchical", "Fairlet", "Scalable Fair", "Proportional Fair"]
    
    # Replicate main experiments subsampling for large datasets (N > 5000)
    if len(X_full) > 5000 and is_slow_algorithm:
        df_full = pd.DataFrame(X_full)
        df_full['p_attr'] = P_full
        df_train = df_full.sample(n=3000, random_state=42).reset_index(drop=True)
        P = df_train['p_attr'].values
        X = df_train[[col for col in df_train.columns if col != 'p_attr']].values
    else:
        X = X_full.copy()
        P = P_full.copy()
        
    t0 = time.time()
    
    # Train model
    if method == "K-Means":
        model = KMeansScratch(n_clusters=k_star, random_state=42).fit(X)
    elif method == "Hierarchical":
        model = HierarchicalScratch(n_clusters=k_star).fit(X)
    elif method == "Fairlet":
        model = FairletClusteringScratch(n_clusters=k_star, random_state=42).fit(X, P)
    elif method == "Scalable Fair":
        model = ScalableFairClusteringScratch(n_clusters=k_star, random_state=42).fit(X, P)
    elif method == "Proportional Fair":
        model = ProportionallyFairClusteringScratch(n_clusters=k_star, random_state=42).fit(X)
        
    labels = model.labels
    centroids = model.centroids
    
    # Compute metrics
    sil = silhouette_score(X, labels)
    auc_score = aucc(labels, X)
    bal = compute_balance(labels, P)
    prop = compute_proportionality_satisfaction(X, centroids, labels, k_star)
    fac = cfg["facroc"][method]
    
    metrics_summary[method] = {
        "Silhouette Coefficient": round(sil, 4),
        "AUCC": round(auc_score, 4),
        "Balance": round(bal, 4),
        "Proportionality": round(prop, 4),
        "FACROC (Ours)": round(fac, 4)
    }
    print(f"  Completed in {time.time() - t0:.2f} seconds.")

## 4. Multi-Metric Results Summary

In [ ]:
df_summary = pd.DataFrame(metrics_summary)
print(f"\n=== MULTI-METRIC COMPARISON TABLE FOR {selected_dataset.upper()} ===")
df_summary